In [1]:
import pandas as pd

df = pd.read_csv("../datasets/cleaned/startup_info.csv")

print(df.shape)
print("\n")

for col in df.columns:
    print(col)

(50000, 78)


startup_name
city
state
country
sector
sub_sector
founded_year
startup_stage
funding_stage
total_funding
investor_count
founder_names
founder_count
patent_count
technology_domain
employee_count
market_growth_rate
market_size
current_status
source_url
data_source
website
linkedin_url
twitter_url
github_url
crunchbase_url
logo_url
latitude
longitude
industry_keywords
business_model
target_market
b2b_b2c
market_type
pricing_model
revenue_model
industry_growth_rate
industry_risk_score
company_type
registration_type
dpiit_registered
gst_registered
international_presence
number_of_offices
female_founder_count
max_founder_experience
iit_founder
iim_founder
foreign_university_founder
trademark_count
mobile_app_available
web_platform_available
api_available
tam
sam
som
customer_segment_count
international_market_access
latest_funding
funding_round_count
investor_names
top_tier_investor_count
average_round_size
valuation
burn_rate
runway_months
employee_growth_rate
customer_count
c

C:\Users\arote\AppData\Local\Temp\ipykernel_2984\650657059.py:3: DtypeWarning: Columns (1,2,3,4,7,8,11,14,18,27,28,29,30,31,32,33,34,35,38,39,57,60) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../datasets/cleaned/startup_info.csv")


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../datasets/cleaned/startup_info.csv",
    low_memory=False
)

print(df.shape)

(50000, 78)


In [3]:
CURRENT_YEAR = 2026

df["startup_age"] = (
    CURRENT_YEAR - df["founded_year"]
)

df["startup_age"] = (
    df["startup_age"]
    .clip(lower=0)
)

In [4]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df["founder_count_norm"] = scaler.fit_transform(
    df[["founder_count"]]
)

df["experience_norm"] = scaler.fit_transform(
    df[["max_founder_experience"]]
)

df["founder_strength_score"] = (

      0.20 * df["founder_count_norm"]

    + 0.30 * df["experience_norm"]

    + 0.15 * df["iit_founder"]

    + 0.15 * df["iim_founder"]

    + 0.10 * df["foreign_university_founder"]

    + 0.10 * (
        df["female_founder_count"]
        /
        df["founder_count"].replace(0,1)
      )

)

df["founder_strength_score"] *= 100

In [5]:
cols = [
    "total_funding",
    "investor_count",
    "funding_round_count",
    "top_tier_investor_count",
    "valuation"
]

for col in cols:

    df[col+"_norm"] = scaler.fit_transform(
        df[[col]]
    )

df["funding_strength_score"] = (

      0.35 * df["total_funding_norm"]

    + 0.20 * df["investor_count_norm"]

    + 0.15 * df["funding_round_count_norm"]

    + 0.20 * df["top_tier_investor_count_norm"]

    + 0.10 * df["valuation_norm"]

)

df["funding_strength_score"] *= 100

In [6]:
market_cols = [
    "market_size",
    "market_growth_rate",
    "tam",
    "sam",
    "som"
]

for col in market_cols:

    df[col+"_norm"] = scaler.fit_transform(
        df[[col]]
    )

df["market_opportunity_score"] = (

      0.30 * df["market_size_norm"]

    + 0.25 * df["market_growth_rate_norm"]

    + 0.20 * df["tam_norm"]

    + 0.15 * df["sam_norm"]

    + 0.10 * df["som_norm"]

)

df["market_opportunity_score"] *= 100

In [7]:
growth_cols = [

    "employee_growth_rate",
    "customer_growth_rate",
    "revenue_growth_rate",
    "monthly_active_users",
    "retention_rate"
]

for col in growth_cols:

    df[col+"_norm"] = scaler.fit_transform(
        df[[col]]
    )

df["growth_score"] = (

      0.20 * df["employee_growth_rate_norm"]

    + 0.25 * df["customer_growth_rate_norm"]

    + 0.25 * df["revenue_growth_rate_norm"]

    + 0.15 * df["monthly_active_users_norm"]

    + 0.15 * df["retention_rate_norm"]

)

df["growth_score"] *= 100

In [8]:
df["startup_health_score"] = (

      0.25 * df["founder_strength_score"]

    + 0.25 * df["funding_strength_score"]

    + 0.20 * df["market_opportunity_score"]

    + 0.20 * df["growth_score"]

    + 0.10 * (
        df["patent_count"]
        /
        df["patent_count"].max()
      ) * 100

)

df["startup_health_score"] = (
    df["startup_health_score"]
    .clip(0,100)
)

In [10]:
df.to_csv(
    "../datasets/cleaned/startup_info.csv",
    index=False
)

print("startup_info.csv Updated Successfully ✅")
print("New Shape:", df.shape)

startup_info.csv Updated Successfully ✅
New Shape: (50000, 101)


In [11]:
new_cols = [

    "startup_age",

    "founder_strength_score",

    "funding_strength_score",

    "market_opportunity_score",

    "growth_score",

    "startup_health_score"
]

print(df[new_cols].head())

   startup_age  founder_strength_score  funding_strength_score  \
0            8               20.060060               32.255275   
1           12               10.540541               17.635687   
2           18               10.510511               38.056868   
3            7               15.090090               13.176755   
4            5               20.210210               32.472872   

   market_opportunity_score  growth_score  startup_health_score  
0                 49.831812      2.759019             23.667000  
1                 35.221591     10.653752             16.389126  
2                 38.203700     13.598091             22.542203  
3                 32.883519     12.032233             16.069862  
4                 30.508111     13.339020             22.120197  


In [12]:
cols_to_drop = [

    "founder_count_norm",
    "experience_norm",

    "total_funding_norm",
    "investor_count_norm",
    "funding_round_count_norm",
    "top_tier_investor_count_norm",
    "valuation_norm",

    "market_size_norm",
    "market_growth_rate_norm",
    "tam_norm",
    "sam_norm",
    "som_norm",

    "employee_growth_rate_norm",
    "customer_growth_rate_norm",
    "revenue_growth_rate_norm",
    "monthly_active_users_norm",
    "retention_rate_norm"
]

df.drop(columns=cols_to_drop, inplace=True)

In [13]:
df.to_csv(
    "../datasets/cleaned/startup_info.csv",
    index=False
)